In [ ]:

from src.raw_data import load_raw_data
from src.clean_data import load_cleaned_data
from src.x_and_y import make_x_and_y, train_test_split_xy
from src.model import train_logreg, train_boosted, print_metrics
from src.scrape import *
from src.inference import *

from src.odds import *

from IPython.display import display
import pandas as pd

def printer(df_dict: dict):
    
    for name, df in df_dict.items():
        print(name, df.shape)
        display(df.head(5))

def main():

    #get raw data, 5 dataframes from github user's scrape of ufc websites
    #so data is included, no external download, fully intergrated into pipeline
    raw = load_raw_data()
    print("STEP 1 RAW DATA\n")
    printer(raw)
    
    #clean all data and make multiple useful utility dataframes, as shown
    cleaned = load_cleaned_data(raw)
    print("\nSTEP 2 CLEANED\n")
    printer(cleaned)

    TRAIN_SIZE = 0.8

    #get x, y and x_train, x_test, y_train, y_test from cleaned["full_sym"]
    print("\nSTEP 3 XY, XY_SPLITS\n")
    xy, xy_splits = make_x_and_y(cleaned["full_sym"], train_size=TRAIN_SIZE)
    printer(xy)
    printer(xy_splits)
    
    #trained both logistic regression and boosted decision tree models and giving info/metrics
    print("\nSTEP 4 MODELS\n")
    model_logreg, model_logreg_metrics = train_logreg(xy_splits)
    model_boosted, model_boosted_metrics = train_boosted(xy_splits)
    print_metrics("LOGREG", model_logreg, model_logreg_metrics)
    print("\n")
    print_metrics("BOOSTED", model_boosted, model_boosted_metrics)

    #showing most productive features for determining if a fighter wins/loses
    print("\nSTEP 4.5 ODDS RATIOS\n")
    odds_df = get_logreg_odds_ratios(
        model_logreg,
        xy_splits["x_train"].columns
    )
    display(odds_df.head(5))
    display(odds_df.tail(5))


    #scraped upcoming event info to get all future fights for inference, 
    #pulled from http://ufcstats.com/statistics/events/upcoming
    print("\nSTEP 5 SCRAPE UPCOMING EVENT INFO\n")
    events_final = get_upcoming_fights_grouped()
    for d in events_final:
        print()
        for k,v in d.items():
            print(f"{k}: {v}")

    #use scraped infor to predict all fights with logreg model as it is interpretable, performs well
    print("\nSTEP 6 PREDICT ALL UPCOMING FIGHTS\n")
    predictions = predict_upcoming_events(events_final, cleaned, model_logreg, xy_splits["x_train"].columns)
    print_predictions(predictions)

    #storing predictions in dataframe for clean access, potential analytics
    print("\nSTEP 7 STORE PREDICTIONS IN DB\n")
    predictions_df = predictions_to_df(predictions)
    display(predictions_df.head(10))


main()


STEP 1 RAW DATA

event_details (770, 4)


,event,url,date,location
0,UFC Fight Night: Burns vs. Malott,http://ufcstats.com/event-details/c3ac8d0da7b0...,"April 18, 2026","Winnipeg, Manitoba, Canada"
1,UFC 327: Prochazka vs. Ulberg,http://ufcstats.com/event-details/f3eb664db7fb...,"April 11, 2026","Miami, Florida, USA"
2,UFC Fight Night: Moicano vs. Duncan,http://ufcstats.com/event-details/9a70f67ad218...,"April 04, 2026","Las Vegas, Nevada, USA"
3,UFC Fight Night: Adesanya vs. Pyfer,http://ufcstats.com/event-details/5c38639f860a...,"March 28, 2026","Seattle, Washington, USA"
4,UFC Fight Night: Evloev vs. Murphy,http://ufcstats.com/event-details/69108cb8b32e...,"March 21, 2026","London, England, United Kingdom"


fight_details (8649, 3)


,event,bout,url
0,UFC Fight Night: Burns vs. Malott,Gilbert Burns vs. Mike Malott,http://ufcstats.com/fight-details/32054bf2b36b...
1,UFC Fight Night: Burns vs. Malott,Kyler Phillips vs. Charles Jourdain,http://ufcstats.com/fight-details/b5299e5b9460...
2,UFC Fight Night: Burns vs. Malott,Mandel Nallo vs. Jai Herbert,http://ufcstats.com/fight-details/e644218737c1...
3,UFC Fight Night: Burns vs. Malott,Jasmine Jasudavicius vs. Karine Silva,http://ufcstats.com/fight-details/5a209522211d...
4,UFC Fight Night: Burns vs. Malott,Thiago Moises vs. Gauge Young,http://ufcstats.com/fight-details/a661c0bee7c8...


results (8649, 11)


...


STEP 4 MODELS

LOGREG:

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('clf', LogisticRegression(max_iter=5000))])

train accuracy: 0.6037508602890571
test  accuracy: 0.6159669649002064
train roc_auc: 0.6389828351906419
test  roc_auc: 0.6671935369733645
fraction close calls: 0.3262216104611149


BOOSTED:

Pipeline(steps=[('imputer', SimpleImputer(strategy='median')),
                ('clf',
                 HistGradientBoostingClassifier(l2_regularization=5.0,
                                                learning_rate=0.03, max_depth=3,
                                                max_iter=1200,
                                                min_samples_leaf=50,
                                                random_state=42))])

train accuracy: 0.6163110805230557
test  accuracy: 0.6025464556090847
train roc_auc: 0.6611656964090245
test  roc_auc: 0.6556773867485408
fraction close calls: 0.37611837577

,feature,coefficient,odds_ratio
21,delta_wins_entering,0.261067,1.298315
10,delta_five_round_rate_entering,0.228027,1.256120
20,delta_win_rate_entering,0.170392,1.185769
3,delta_avg_fight_time_entering,0.122521,1.130342
5,delta_fights_entering,0.086330,1.090166


,feature,coefficient,odds_ratio
12,delta_ko_losses_entering,-0.036556,0.964105
14,delta_ko_wins_entering,-0.054030,0.947404
9,delta_five_round_fights_entering,-0.149995,0.860713
15,delta_losses_entering,-0.188998,0.827788
2,delta_age_at_fight,-0.314207,0.730368



STEP 5 SCRAPE UPCOMING EVENT INFO


name: UFC Fight Night: Sterling vs. Zalal
url: http://ufcstats.com/event-details/e60d773a0a42048a
date_str: April 25, 2026
date: 2026-04-25 00:00:00
fights: [{'fighter_a': 'Aljamain Sterling', 'fighter_b': 'Youssef Zalal'}, {'fighter_a': 'Norma Dumont', 'fighter_b': 'Joselyne Edwards'}, {'fighter_a': 'Rafa Garcia', 'fighter_b': 'Alexander Hernandez'}, {'fighter_a': 'Davey Grant', 'fighter_b': 'Juan Adrian Martinetti'}, {'fighter_a': 'Montel Jackson', 'fighter_b': 'Raoni Barcelos'}, {'fighter_a': 'Marcus Buchecha', 'fighter_b': 'Ryan Spann'}, {'fighter_a': 'Rodolfo Vieira', 'fighter_b': 'Eric McConico'}, {'fighter_a': 'Jackson McVey', 'fighter_b': 'Sedriques Dumas'}, {'fighter_a': 'Mayra Bueno Silva', 'fighter_b': 'Michelle Montague'}, {'fighter_a': 'Jafel Filho', 'fighter_b': 'Lucas Rocha'}, {'fighter_a': 'Talita Alencar', 'fighter_b': 'Julia Polastri'}]

name: UFC Fight Night: Della Maddalena vs. Prates
url: http://ufcstats.com/event-details/872b01

,event,date,fighter_a,fighter_b,prob_fighter_a_wins,prob_fighter_b_wins,predicted_winner
0,UFC Fight Night: Sterling vs. Zalal,2026-04-25,Aljamain Sterling,Youssef Zalal,0.519933,0.480067,Aljamain Sterling
1,UFC Fight Night: Sterling vs. Zalal,2026-04-25,Norma Dumont,Joselyne Edwards,NaN,NaN,None
2,UFC Fight Night: Sterling vs. Zalal,2026-04-25,Rafa Garcia,Alexander Hernandez,0.531725,0.468275,Rafa Garcia
3,UFC Fight Night: Sterling vs. Zalal,2026-04-25,Davey Grant,Juan Adrian Martinetti,NaN,NaN,None
4,UFC Fight Night: Sterling vs. Zalal,2026-04-25,Montel Jackson,Raoni Barcelos,0.622507,0.377493,Montel Jackson
5,UFC Fight Night: Sterling vs. Zalal,2026-04-25,Marcus Buchecha,Ryan Spann,0.335311,0.664689,Ryan Spann
6,UFC Fight Night: Sterling vs. Zalal,2026-04-25,Rodolfo Vieira,Eric McConico,0.601618,0.398382,Rodolfo Vieira
7,UFC Fight Night: Sterling vs. Zalal,2026-04-25,Jackson McVey,Sedriques Dumas,0.477670,0.522330,Sedriques Dumas
8,UFC Fight Night: Sterling vs. Zalal,2026-04-25,Mayra Bueno Silva,Michelle Montague,NaN,NaN,None
9,UFC Fight Night: Sterling vs. Zalal,2026-04-25,Jafel Filho,Lucas Rocha,0.530194,0.469806,Jafel Filho
